# EDA - Swedavia FlightInfo API + Wikipedia

In [2]:
import sys, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# make the repo root importable 
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "explorations" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))


from data_extract_load.config import SWEDAVIA_AIRPORTS
from data_extract_load.swedavia_api import get_flights, last_n_dates
from data_extract_load.wikipedia import get_swedish_airports, get_airline_codes

pd.set_option("display.max_columns", 100)
dates = last_n_dates()
dates

['2026-10-15', '2026-10-14', '2026-10-13']

## 1. One raw response

In [3]:
payload = get_flights("arrivals", "ARN", dates[0])
print(payload.keys())
print("number of flights:", payload.get("numberOfFlights"))

dict_keys(['to', 'numberOfFlights', 'flights'])
number of flights: 346


## one flight example

In [4]:
print(json.dumps(payload["flights"][0], indent=2, ensure_ascii=False))

{
  "flightId": "FR4464",
  "departureAirportSwedish": "Birmingham",
  "departureAirportEnglish": "Birmingham",
  "airlineOperator": {
    "iata": "FR",
    "icao": "RYR",
    "name": "Ryanair Ltd"
  },
  "arrivalTime": {
    "scheduledUtc": "2026-10-15T08:45:00Z"
  },
  "locationAndStatus": {
    "terminal": "T5",
    "flightLegStatus": "DEL",
    "flightLegStatusSwedish": "Borttagen",
    "flightLegStatusEnglish": "Deleted"
  },
  "baggage": {},
  "codeShareData": [],
  "flightLegIdentifier": {
    "callsign": "RYR4464",
    "flightId": "FR4464",
    "flightDepartureDateUtc": "2026-10-15T00:00:00Z",
    "departureAirportIata": "BHX",
    "arrivalAirportIata": "ARN",
    "departureAirportIcao": "EGBB",
    "arrivalAirportIcao": "ESSA"
  },
  "remarksEnglish": [],
  "remarksSwedish": [],
  "viaDestinations": [],
  "diIndicator": "I"
}


## 2. Flatten to a DataFrame, dlt does the same

In [5]:
arr_one = pd.json_normalize(payload["flights"])
arr_one.info()

<class 'pandas.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype 
---  ------                                      --------------  ----- 
 0   flightId                                    346 non-null    str   
 1   departureAirportSwedish                     346 non-null    str   
 2   departureAirportEnglish                     346 non-null    str   
 3   codeShareData                               346 non-null    object
 4   remarksEnglish                              346 non-null    object
 5   remarksSwedish                              346 non-null    object
 6   viaDestinations                             346 non-null    object
 7   diIndicator                                 346 non-null    str   
 8   airlineOperator.iata                        346 non-null    str   
 9   airlineOperator.icao                        346 non-null    str   
 10  airlineOperator.name                 

## 3b. Explore the full tables

In [11]:
def fetch_all(direction: str) -> pd.DataFrame:
    frames = []
    for airport in SWEDAVIA_AIRPORTS:
        for d in dates:
            flights = get_flights(direction, airport, d).get("flights", [])
            df = pd.json_normalize(flights)
            df["query_airport"], df["query_date"] = airport, d
            frames.append(df)
    return pd.concat(frames, ignore_index=True)

arrivals = fetch_all("arrivals")
departures = fetch_all("departures")
arrivals.shape, departures.shape

((1557, 34), (1558, 40))

In [12]:
def show_table(df: pd.DataFrame, n: int = 15, columns: list[str] | None = None,
               transpose: bool = False, sample: bool = False):
    if columns:
        df = df[[c for c in df.columns if any(word.lower() in c.lower() for word in columns)]]
    rows = df.sample(min(n, len(df)), random_state=42) if sample else df.head(n)
    print(f"{len(df):,} rows x {df.shape[1]} columns  (showing {len(rows)} rows)")
    with pd.option_context("display.max_columns", None, "display.width", None,
                           "display.max_colwidth", 60, "display.max_rows", 500):
        display(rows.T if transpose else rows)

In [13]:
# all columns, first 15 arrivals
show_table(arrivals)

1,557 rows x 34 columns  (showing 15 rows)


,flightId,departureAirportSwedish,departureAirportEnglish,codeShareData,remarksEnglish,remarksSwedish,viaDestinations,diIndicator,airlineOperator.iata,airlineOperator.icao,airlineOperator.name,arrivalTime.scheduledUtc,locationAndStatus.terminal,locationAndStatus.flightLegStatus,locationAndStatus.flightLegStatusSwedish,locationAndStatus.flightLegStatusEnglish,flightLegIdentifier.callsign,flightLegIdentifier.flightId,flightLegIdentifier.flightDepartureDateUtc,flightLegIdentifier.departureAirportIata,flightLegIdentifier.arrivalAirportIata,flightLegIdentifier.departureAirportIcao,flightLegIdentifier.arrivalAirportIcao,baggage.baggageClaimUnit,flightLegIdentifier.aircraftRegistration,locationAndStatus.gate,arrivalTime.estimatedUtc,flightLegIdentifier.ssrCode,arrivalTime.actualUtc,query_airport,query_date,baggage.firstBagUtc,baggage.lastBagUtc,baggage.estimatedFirstBagUtc
0,FR4464,Birmingham,Birmingham,[],[],[],[],I,FR,RYR,Ryanair Ltd,2026-10-15T08:45:00Z,T5,DEL,Borttagen,Deleted,RYR4464,FR4464,2026-10-15T00:00:00Z,BHX,ARN,EGBB,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
1,FR3008,Dublin,Dublin,[],[],[],[],I,FR,RYR,Ryanair Ltd,2026-10-15T09:30:00Z,T5,DEL,Borttagen,Deleted,RYR3008,FR3008,2026-10-15T00:00:00Z,DUB,ARN,EIDW,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
2,FR971,Zadar,Zadar,[],[],[],[],S,FR,RYR,Ryanair Ltd,2026-10-15T09:50:00Z,T5,DEL,Borttagen,Deleted,RYR971,FR971,2026-10-15T00:00:00Z,ZAD,ARN,LDZD,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
3,FR9447,Manchester,Manchester,[],[],[],[],I,FR,RYR,Ryanair Ltd,2026-10-15T21:00:00Z,T5,DEL,Borttagen,Deleted,RYR9447,FR9447,2026-10-15T00:00:00Z,MAN,ARN,EGCC,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
4,FR6833,Malaga,Malaga,[],[],[],[],S,FR,RYR,Ryanair Ltd,2026-10-15T21:45:00Z,T5,DEL,Borttagen,Deleted,RYR6833,FR6833,2026-10-15T00:00:00Z,AGP,ARN,LEMG,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
5,DE4409,Frankfurt,Frankfurt,[],[],[],[],S,DE,CFG,CONDOR FLUGDIENST,2026-10-15T08:00:00Z,T5,DEL,Borttagen,Deleted,CFG4409,DE4409,2026-10-15T00:00:00Z,FRA,ARN,EDDF,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
6,DE4405,Frankfurt,Frankfurt,[],[],[],[],S,DE,CFG,CONDOR FLUGDIENST,2026-10-15T14:10:00Z,T5,DEL,Borttagen,Deleted,CFG4405,DE4405,2026-10-15T00:00:00Z,FRA,ARN,EDDF,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
7,W62407,Budapest,Budapest,[],[],[],[],S,W6,WZZ,Wizz Air,2026-10-15T06:25:00Z,T5,DEL,Borttagen,Deleted,WZZ2407,W62407,2026-10-15T00:00:00Z,BUD,ARN,LHBP,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
8,W62403,Budapest,Budapest,[],[],[],[],S,W6,WZZ,Wizz Air,2026-10-15T18:20:00Z,T5,DEL,Borttagen,Deleted,WZZ56,W62403,2026-10-15T00:00:00Z,BUD,ARN,LHBP,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN
9,W64809,Skopje,Skopje,[],[],[],[],I,W6,WZZ,Wizz Air,2026-10-15T07:00:00Z,T5,DEL,Borttagen,Deleted,WZZ1459,W64809,2026-10-15T00:00:00Z,SKP,ARN,LWSK,ESSA,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN


In [14]:
# all columns, first 15 departures
show_table(departures)

1,558 rows x 40 columns  (showing 15 rows)


,flightId,arrivalAirportSwedish,arrivalAirportEnglish,codeShareData,viaDestinations,remarksEnglish,remarksSwedish,diIndicator,airlineOperator.iata,airlineOperator.icao,airlineOperator.name,departureTime.scheduledUtc,locationAndStatus.terminal,locationAndStatus.flightLegStatus,locationAndStatus.flightLegStatusSwedish,locationAndStatus.flightLegStatusEnglish,flightLegIdentifier.callsign,flightLegIdentifier.flightId,flightLegIdentifier.flightDepartureDateUtc,flightLegIdentifier.departureAirportIata,flightLegIdentifier.arrivalAirportIata,flightLegIdentifier.departureAirportIcao,flightLegIdentifier.arrivalAirportIcao,checkIn.checkInDeskFrom,checkIn.checkInDeskTo,flightLegIdentifier.aircraftRegistration,locationAndStatus.gate,departureTime.estimatedUtc,flightLegIdentifier.ssrCode,query_airport,query_date,departureTime.actualUtc,locationAndStatus.gateAction,locationAndStatus.gateActionSwedish,locationAndStatus.gateActionEnglish,locationAndStatus.gateCloseUtc,checkIn.checkInStatus,checkIn.checkInStatusSwedish,checkIn.checkInStatusEnglish,locationAndStatus.gateOpenUtc
0,FR970,Zadar,Zadar,[],[],[],[],S,FR,RYR,Ryanair Ltd,2026-10-15T04:00:00Z,T5,DEL,Borttagen,Deleted,RYR93AX,FR970,2026-10-15T00:00:00Z,ARN,ZAD,ESSA,LDZD,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FR4465,Birmingham,Birmingham,[],[],[],[],I,FR,RYR,Ryanair Ltd,2026-10-15T09:10:00Z,T5,DEL,Borttagen,Deleted,RYR4465,FR4465,2026-10-15T00:00:00Z,ARN,BHX,ESSA,EGBB,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FR3009,Dublin,Dublin,[],[],[],[],I,FR,RYR,Ryanair Ltd,2026-10-15T09:55:00Z,T5,DEL,Borttagen,Deleted,RYR3009,FR3009,2026-10-15T00:00:00Z,ARN,DUB,ESSA,EIDW,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FR4116,Gdansk,Gdansk,[],[],[],[],S,FR,RYR,Ryanair Ltd,2026-10-15T20:45:00Z,T5,DEL,Borttagen,Deleted,RYR4116,FR4116,2026-10-15T00:00:00Z,ARN,GDN,ESSA,EPGD,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FR9448,Manchester,Manchester,[],[],[],[],I,FR,RYR,Ryanair Ltd,2026-10-15T21:25:00Z,T5,DEL,Borttagen,Deleted,RYR9448,FR9448,2026-10-15T00:00:00Z,ARN,MAN,ESSA,EGCC,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,DE4404,Frankfurt,Frankfurt,[],[],[],[],S,DE,CFG,CONDOR FLUGDIENST,2026-10-15T05:50:00Z,T5,DEL,Borttagen,Deleted,CFG4404,DE4404,2026-10-15T00:00:00Z,ARN,FRA,ESSA,EDDF,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,DE4408,Frankfurt,Frankfurt,[],[],[],[],S,DE,CFG,CONDOR FLUGDIENST,2026-10-15T08:50:00Z,T5,DEL,Borttagen,Deleted,CFG4408,DE4408,2026-10-15T00:00:00Z,ARN,FRA,ESSA,EDDF,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DE4406,Frankfurt,Frankfurt,[],[],[],[],S,DE,CFG,CONDOR FLUGDIENST,2026-10-15T15:00:00Z,T5,DEL,Borttagen,Deleted,CFG4406,DE4406,2026-10-15T00:00:00Z,ARN,FRA,ESSA,EDDF,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,W62408,Budapest,Budapest,[],[],[],[],S,W6,WZZ,Wizz Air,2026-10-15T07:00:00Z,T5,DEL,Borttagen,Deleted,WZZ2408,W62408,2026-10-15T00:00:00Z,ARN,BUD,ESSA,LHBP,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,W62404,Budapest,Budapest,[],[],[],[],S,W6,WZZ,Wizz Air,2026-10-15T18:55:00Z,T5,DEL,Borttagen,Deleted,WZZ877,W62404,2026-10-15T00:00:00Z,ARN,BUD,ESSA,LHBP,NaN,NaN,NaN,NaN,NaN,NaN,ARN,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
show_table(arrivals, n=5, transpose=True, sample=True)

1,557 rows x 34 columns  (showing 5 rows)


,1496,168,353,468,1410
flightId,OJ804,SK580,DE4405,SK2546,SK008
departureAirportSwedish,Stockholm BMA,Paris CDG,Frankfurt,Edinburgh,Stockholm ARN
departureAirportEnglish,Stockholm BMA,Paris CDG,Frankfurt,Edinburgh,Stockholm ARN
codeShareData,[],"[AF4558, AM7814, LY9751]",[],[],"[KL2908, LO5411]"
remarksEnglish,[],[],[],"[{'text': 'Landed 12:23', 'indicator': 'NEUTRAL'}, {'tex...","[{'text': 'Landed 13:07', 'indicator': 'NEUTRAL'}, {'tex..."
remarksSwedish,[],[],[],"[{'text': 'Landat 12:23', 'indicator': 'NEUTRAL'}, {'tex...","[{'text': 'Landat 13:07', 'indicator': 'NEUTRAL'}, {'tex..."
viaDestinations,[],[],[],[],[]
diIndicator,D,S,S,I,D
airlineOperator.iata,OJ,SK,DE,SK,SK
airlineOperator.icao,NYX,SAS,CFG,SAS,SAS


In [16]:
show_table(departures, n=10, columns=["flightId", "airline", "time", "status"])

1,558 rows x 21 columns  (showing 10 rows)


,flightId,airlineOperator.iata,airlineOperator.icao,airlineOperator.name,departureTime.scheduledUtc,locationAndStatus.terminal,locationAndStatus.flightLegStatus,locationAndStatus.flightLegStatusSwedish,locationAndStatus.flightLegStatusEnglish,flightLegIdentifier.flightId,locationAndStatus.gate,departureTime.estimatedUtc,departureTime.actualUtc,locationAndStatus.gateAction,locationAndStatus.gateActionSwedish,locationAndStatus.gateActionEnglish,locationAndStatus.gateCloseUtc,checkIn.checkInStatus,checkIn.checkInStatusSwedish,checkIn.checkInStatusEnglish,locationAndStatus.gateOpenUtc
0,FR970,FR,RYR,Ryanair Ltd,2026-10-15T04:00:00Z,T5,DEL,Borttagen,Deleted,FR970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FR4465,FR,RYR,Ryanair Ltd,2026-10-15T09:10:00Z,T5,DEL,Borttagen,Deleted,FR4465,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FR3009,FR,RYR,Ryanair Ltd,2026-10-15T09:55:00Z,T5,DEL,Borttagen,Deleted,FR3009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FR4116,FR,RYR,Ryanair Ltd,2026-10-15T20:45:00Z,T5,DEL,Borttagen,Deleted,FR4116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FR9448,FR,RYR,Ryanair Ltd,2026-10-15T21:25:00Z,T5,DEL,Borttagen,Deleted,FR9448,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,DE4404,DE,CFG,CONDOR FLUGDIENST,2026-10-15T05:50:00Z,T5,DEL,Borttagen,Deleted,DE4404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,DE4408,DE,CFG,CONDOR FLUGDIENST,2026-10-15T08:50:00Z,T5,DEL,Borttagen,Deleted,DE4408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DE4406,DE,CFG,CONDOR FLUGDIENST,2026-10-15T15:00:00Z,T5,DEL,Borttagen,Deleted,DE4406,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,W62408,W6,WZZ,Wizz Air,2026-10-15T07:00:00Z,T5,DEL,Borttagen,Deleted,W62408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,W62404,W6,WZZ,Wizz Air,2026-10-15T18:55:00Z,T5,DEL,Borttagen,Deleted,W62404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# filter rows first, e.g. departures from Bromma on the first date
show_table(departures[(departures["query_airport"] == "BMA") & (departures["query_date"] == dates[0])], n=10)

22 rows x 40 columns  (showing 10 rows)


,flightId,arrivalAirportSwedish,arrivalAirportEnglish,codeShareData,viaDestinations,remarksEnglish,remarksSwedish,diIndicator,airlineOperator.iata,airlineOperator.icao,airlineOperator.name,departureTime.scheduledUtc,locationAndStatus.terminal,locationAndStatus.flightLegStatus,locationAndStatus.flightLegStatusSwedish,locationAndStatus.flightLegStatusEnglish,flightLegIdentifier.callsign,flightLegIdentifier.flightId,flightLegIdentifier.flightDepartureDateUtc,flightLegIdentifier.departureAirportIata,flightLegIdentifier.arrivalAirportIata,flightLegIdentifier.departureAirportIcao,flightLegIdentifier.arrivalAirportIcao,checkIn.checkInDeskFrom,checkIn.checkInDeskTo,flightLegIdentifier.aircraftRegistration,locationAndStatus.gate,departureTime.estimatedUtc,flightLegIdentifier.ssrCode,query_airport,query_date,departureTime.actualUtc,locationAndStatus.gateAction,locationAndStatus.gateActionSwedish,locationAndStatus.gateActionEnglish,locationAndStatus.gateCloseUtc,checkIn.checkInStatus,checkIn.checkInStatusSwedish,checkIn.checkInStatusEnglish,locationAndStatus.gateOpenUtc
1033,FRO671,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T06:10:00Z,T1,DEL,Borttagen,Deleted,FRO671,FRO671,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1034,FRO673,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T07:15:00Z,T1,DEL,Borttagen,Deleted,FRO673,FRO673,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1035,FRO675,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T09:15:00Z,T1,DEL,Borttagen,Deleted,FRO675,FRO675,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1036,FRO679,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T14:00:00Z,T1,DEL,Borttagen,Deleted,FRO679,FRO679,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1037,FRO681,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T15:30:00Z,T1,DEL,Borttagen,Deleted,FRO681,FRO681,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1038,FRO685,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T16:55:00Z,T1,DEL,Borttagen,Deleted,FRO685,FRO685,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1039,FRO687,Malmö,Malmö,[],[],[],[],D,FRO,FRO,FROSTAIR AIR APS,2026-10-15T18:30:00Z,T1,DEL,Borttagen,Deleted,FRO687,FRO687,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1040,OJ804,Visby,Visby,[],[],[],[],D,OJ,NYX,Nyxair,2026-10-15T08:00:00Z,T1,DEL,Borttagen,Deleted,NYX804,OJ804,2026-10-15T00:00:00Z,BMA,VBY,ESSB,ESSV,NaN,NaN,NaN,NaN,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1041,OJ802,Visby,Visby,[],[],[],[],D,OJ,NYX,Nyxair,2026-10-15T06:30:00Z,T1,SCH,Planerad,Scheduled,NYX802,OJ802,2026-10-15T00:00:00Z,BMA,VBY,ESSB,ESSV,3.0,3.0,ESNTB,J,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1042,FT679,Malmö,Malmö,[],[],[],[],D,FT,FT,FROST Air,2026-10-15T14:00:00Z,T1,SCH,Planerad,Scheduled,FRO679,FT679,2026-10-15T00:00:00Z,BMA,MMX,ESSB,ESMS,2.0,2.0,OYFSF,M,NaN,NaN,BMA,2026-10-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
